In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

print("=" * 70)
print("   AI-BASED HEART DISEASE PREDICTION & RISK ANALYSIS SYSTEM")
print("=" * 70)


# 2. LOAD DATASET
file_path = "heart_disease_data (1).csv"

try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print("\nCSV file not found in the current folder.")
    print("Please place 'heart_disease_data (1).csv' in the same folder as this notebook.")
    raise

print("\nDataset loaded successfully!")
print("Dataset Shape:", df.shape)


# 3. BASIC DATA INFORMATION
print("\n--- FIRST 5 RECORDS ---")
display(df.head())

print("\n--- DATASET INFORMATION ---")
print(df.info())

print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- DUPLICATE RECORDS ---")
print(df.duplicated().sum())


# 4. DATA CLEANING
df = df.drop_duplicates()
df = df.dropna()

print("\nShape after cleaning:", df.shape)


# 5. TARGET DISTRIBUTION
print("\n--- TARGET DISTRIBUTION ---")
print(df["target"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x="target", data=df)
plt.title("Heart Disease Distribution")
plt.xlabel("Target (0 = No Disease, 1 = Disease)")
plt.ylabel("Number of Patients")
plt.show()


# 6. EXPLORATORY DATA ANALYSIS

# Age distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="age", hue="target", kde=True)
plt.title("Age Distribution by Heart Disease Status")
plt.show()

# Gender distribution
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="sex", hue="target")
plt.title("Heart Disease Distribution by Gender")
plt.xlabel("Sex")
plt.ylabel("Number of Patients")
plt.show()

# Chest pain
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="cp", hue="target")
plt.title("Chest Pain Type vs Heart Disease")
plt.xlabel("Chest Pain Type")
plt.ylabel("Number of Patients")
plt.show()

# Maximum heart rate
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="target", y="thalach")
plt.title("Maximum Heart Rate vs Heart Disease")
plt.show()

# Cholesterol
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="target", y="chol")
plt.title("Cholesterol vs Heart Disease")
plt.show()


# 7. CORRELATION HEATMAP
plt.figure(figsize=(14, 10))
sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Feature Correlation Heatmap")
plt.show()


# 8. SEPARATE FEATURES AND TARGET
X = df.drop("target", axis=1)
y = df["target"]

print("\nFeatures:", list(X.columns))
print("Number of Features:", X.shape[1])


# 9. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# 10. FEATURE SCALING
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# 11. CREATE MODELS
models = {
    "Logistic Regression": (
        LogisticRegression(random_state=42),
        True
    ),
    
    "Decision Tree": (
        DecisionTreeClassifier(
            random_state=42,
            max_depth=5
        ),
        False
    ),
    
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=200,
            max_depth=7,
            random_state=42
        ),
        False
    ),
    
    "KNN": (
        KNeighborsClassifier(n_neighbors=5),
        True
    ),
    
    "SVM": (
        SVC(
            probability=True,
            random_state=42
        ),
        True
    )
}


# 12. TRAIN AND EVALUATE MODELS
results = []
trained_models = {}

for name, (model, scaled) in models.items():

    if scaled:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)
        probabilities = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)[:, 1]

    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1 Score": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })


# 13. MODEL COMPARISON
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("                 MODEL COMPARISON")
print("=" * 70)

display(
    results_df.sort_values(
        by="F1 Score",
        ascending=False
    ).round(4)
)


# 14. MODEL COMPARISON GRAPH
plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_df,
    x="Model",
    y="Accuracy"
)

plt.title("Machine Learning Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.show()


# 15. SELECT BEST MODEL USING F1 SCORE
best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "Model"
]

best_model = trained_models[best_model_name]

print("\nBEST MODEL:", best_model_name)

best_scaled = models[best_model_name][1]

if best_scaled:
    best_predictions = best_model.predict(X_test_scaled)
    best_probabilities = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    best_predictions = best_model.predict(X_test)
    best_probabilities = best_model.predict_proba(X_test)[:, 1]


# 16. BEST MODEL REPORT
print("\n" + "=" * 70)
print("                 BEST MODEL REPORT")
print("=" * 70)

print("\nAccuracy:",
      round(accuracy_score(y_test, best_predictions), 4))

print("Precision:",
      round(precision_score(y_test, best_predictions), 4))

print("Recall:",
      round(recall_score(y_test, best_predictions), 4))

print("F1 Score:",
      round(f1_score(y_test, best_predictions), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test, best_probabilities), 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        best_predictions,
        target_names=["No Disease", "Disease"]
    )
)


# 17. CONFUSION MATRIX
cm = confusion_matrix(
    y_test,
    best_predictions
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()


# 18. ROC CURVE
fpr, tpr, thresholds = roc_curve(
    y_test,
    best_probabilities
)

auc = roc_auc_score(
    y_test,
    best_probabilities
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"{best_model_name} (AUC = {auc:.2f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


# 19. FEATURE IMPORTANCE
# Random Forest is used here specifically to analyze feature importance.
rf_model = trained_models["Random Forest"]

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n" + "=" * 70)
print("                 FEATURE IMPORTANCE")
print("=" * 70)

display(importance.round(4))

plt.figure(figsize=(10, 7))

sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)

plt.title("Feature Importance - Random Forest")
plt.show()


# 20. PATIENT PREDICTION FUNCTION
def predict_heart_disease(patient_data):

    patient_df = pd.DataFrame(
        [patient_data],
        columns=X.columns
    )

    if best_scaled:
        patient_scaled = scaler.transform(patient_df)

        prediction = best_model.predict(patient_scaled)[0]
        probability = best_model.predict_proba(
            patient_scaled
        )[0][1]

    else:
        prediction = best_model.predict(patient_df)[0]
        probability = best_model.predict_proba(
            patient_df
        )[0][1]

    print("\n" + "=" * 60)
    print("             PATIENT RISK ANALYSIS")
    print("=" * 60)

    print("Prediction:",
          "Heart Disease Detected"
          if prediction == 1
          else "No Heart Disease Detected")

    print("Model Probability:",
          round(probability * 100, 2), "%")

    if probability < 0.30:
        risk = "LOW"
    elif probability < 0.60:
        risk = "MODERATE"
    else:
        risk = "HIGH"

    print("Risk Level:", risk)

    print("\nNOTE: This is an educational machine-learning")
    print("prediction and is NOT a medical diagnosis.")

    return prediction, probability


# 21. EXAMPLE PATIENT
# The values follow the standard UCI/Kaggle heart-disease
# dataset feature order.

example_patient = [
    55,     # age
    1,      # sex
    1,      # cp
    130,    # trestbps
    250,    # chol
    0,      # fbs
    1,      # restecg
    150,    # thalach
    0,      # exang
    1.0,    # oldpeak
    1,      # slope
    1,      # ca
    2       # thal
]

print("\nExample Patient:")
print(pd.DataFrame(
    [example_patient],
    columns=X.columns
))

predict_heart_disease(example_patient)


# ============================================================
# PROJECT COMPLETED
# ============================================================

print("\n" + "=" * 70)
print("              PROJECT COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Dataset Shape:", df.shape)
print("Best Model:", best_model_name)
print("Best F1 Score:",
      round(
          results_df.loc[
              results_df["F1 Score"].idxmax(),
              "F1 Score"
          ],
          4
      ))
print("=" * 70)